In [ ]:
import os
import time
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from pathlib import Path

LOCAL_ROOT = r"/Users/takanohiroshi/Library/CloudStorage/OneDrive-個人用/Documenten/GitHub/FileCopy/"

LOCAL_ROOT = r"C:\Users\tknhi\OneDrive\Documenten\GitHub\FileCopy"

LOCAL_ROOT = r"D:\Users\tknhi\Downloads\FileCopy"

os.chdir(LOCAL_ROOT)
path = os.getcwd()
print(path)

try:
  os.chdir(path + '/Original')
except:
  pass

path = os.getcwd()

SERVICE_ACCOUNT_FILE = LOCAL_ROOT + '/my-project-vscode-452201-249bdb033f8c.json'  # JSONファイルのパス
SCOPES = ["https://www.googleapis.com/auth/drive"]

LOCAL_ROOT = Path.cwd()

DRIVE_ROOT_ID = "0AHJyZSDX6gPHUk9PVA"

MAX_SIZE_MB = 10

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

service = build("drive","v3",credentials=creds)

print(path)


C:\Users\tknhi\OneDrive\Documenten\GitHub\FileCopy
C:\Users\tknhi\OneDrive\Documenten\GitHub\FileCopy\Original


c:\Users\tknhi\myenv\.venv\lib\site-packages\google\api_core\_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.11). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [2]:
# 内部チェック用の記録
debug_info = {
    "last_local_timestamp": None,
    "last_drive_timestamp": None,
    "last_local_size_mb": None,
    "last_max_size_mb": MAX_SIZE_MB,
    "last_drive_search_name": None,
    "last_drive_search_found": None,
    "last_folder_name": None,
    "last_folder_id": None,
    "last_upload_decision": None,
    "error_files": []
}

In [3]:
# Driceにフォルダを作成（存在しなけば）

def get_or_create_folder(name, parent_id):
    try:
        query = (
            f"name='{name}' and mimeType='application/vnd.google-apps.folder'"
            f" and '{parent_id}' in parents"
        )

        results = service.files().list(
            q=query,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True
        ).execute()

        folders = results.get("files", [])
        if folders:
            return folders[0]["id"]

        file_metadata = {
            "name": name,
            "mimeType": "application/vnd.google-apps.folder",
            "parents": [parent_id]
        }

        folder = service.files().create(
            body=file_metadata,
            fields="id",
            supportsAllDrives=True
        ).execute()
        folder_id = folder["id"]

        debug_info["last_folder_name"] = name
        debug_info["last_folder_id"] = folder_id

        return folder_id

    except Exception as e:
        debug_info["error_files"].append(f"[FOLDER] {name} ({e})")
        raise

In [4]:
# Drice上のファイルの更新日時を取得
def get_drive_modified_time(file_id):
    file = service.files().get(
        fileId=file_id,
        fields="modifiedTime",
        supportsAllDrives=True
    ).execute()
    return file["modifiedTime"]

# Drive上に同名のファイルがあるか検索
def find_drive_file(name, parent_id):
    try:
        query = f"name='{name}' and '{parent_id}' in parents"

        results = service.files().list(
            q=query,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True
        ).execute()

        files = results.get("files",[])
        debug_info["last_drive_search_name"] = name
        debug_info["last_drive_search_found"] = bool(files)

        return files[0] if files else None
    
    except Exception as e:
        debug_info["error_files"].append(f"[SEARCH] {name} ({e})")
        return None

# ファイルをアップロード（新しい場合のみ）
def upload_file(local_path, parent_id):
    try:
        size_mb = os.path.getsize(local_path) / (1024*1024)
        debug_info["last_local_size_mb"] = round(size_mb, 3)

        if size_mb > MAX_SIZE_MB:
            debug_info["last_upload_decision"] = "SKIP_SIZE"
            return
        
        name = os.path.basename(local_path)
        local_mtime = os.path.getmtime(local_path)
        debug_info["last_local_timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(local_mtime)) 

        drive_file = find_drive_file(name, parent_id)

        # Driceに同名ファイルがある場合->新しい方だけ反映
        if drive_file:
            drive_mtime_str = get_drive_modified_time(drive_file["id"])
            debug_info["last_drive_timestamp"] = drive_mtime_str

            try:
                drive_mtime = time.mktime(time.strptime(drive_mtime_str, "%Y-%m-%dT%H:%M:%S.%fZ"))
            except ValueError:
                drive_mtime = time.mktime(time.strptime(drive_mtime_str, "%Y-%m-%dT%H:%M:%SZ"))
            
            if drive_mtime >= local_mtime:
                debug_info["last_upload_detecsion"] = "SKIP_DRIVE_NEWER"
                return

            debug_info["last_upload_decision"] = "UPDATE"
            media = MediaFileUpload(local_path, resumable=True)
            service.files().update(
                fileId=drive_file["id"],
                media_body=media,
                supportsAllDrives=True
            ).execute()
            return

        # Driveにファイルがない場合->新規アップロード
        debug_info["last_upload_decision"] = "UPLOAD_NEW"
        media = MediaFileUpload(local_path, resumable=True)
        file_metadata = {"name":name, "parents":[parent_id]}

        service.files().create(
            body=file_metadata,
            media_body=media,
            fields="id",
            supportsAllDrives=True
        ).execute()
        
    except Exception as e:
        debug_info["error_files"].append(f"[FILE] {local_path} ({e})")


# 再起的にフォルダー構造をコピー
def copy_recursive(local_dir, drive_parent_id):
    for item in os.listdir(local_dir):
        local_path = os.path.join(local_dir, item)

        if os.path.isdir(local_path):
            new_drive_folder_id = get_or_create_folder(item, drive_parent_id)
            copy_recursive(local_path, new_drive_folder_id)
        else:
            upload_file(local_path, drive_parent_id)


In [5]:
!cd LOCAL_ROOT
!ls

�w�肳�ꂽ�p�X��������܂���B
'ls' �́A�����R�}���h�܂��͊O���R�}���h�A
����\�ȃv���O�����܂��̓o�b�` �t�@�C���Ƃ��ĔF������Ă��܂���B


In [6]:
print("コピー開始")
copy_recursive(LOCAL_ROOT, DRIVE_ROOT_ID)
print("終了")

print(f"[タイムスタンプ]")
print(f" ローカル： {debug_info['last_local_timestamp']} ")
print(f" Drive側： {debug_info['last_drive_timestamp']}")

print(f"ファイルサイズ")
print(f" ローカル： {debug_info['last_local_size_mb']} MB")
print(f" MAX_SIZE_MB: {debug_info['last_max_size_mb']} MB")

print(f"\n[Drive検索]")
print(f" ファイル名：{debug_info['last_drive_search_name']}")
print(f" 検索結果： {'あり' if debug_info['last_drive_search_found'] else 'なし'}")

print(f"\n[フォルダ作成]")
print(f" フォルダ名： {debug_info['last_folder_name']}")
print(f" Dricve側フォルダID： {debug_info['last_folder_id']}")

print(f"\n[アップロード判定]")
print(f" 判定： {debug_info['last_upload_decision']}")

print(f"\n[エラー件数]")
print(f" {len(debug_info['error_files'])}")
if debug_info["error_files"]:
    print(" 詳細：")
    for e in debug_info['error_files']:
        print(f"- {e}")




コピー開始
終了
[タイムスタンプ]
 ローカル： 2026-11-01 15:23:19 
 Drive側： None
ファイルサイズ
 ローカル： 0.819 MB
 MAX_SIZE_MB: 10 MB

[Drive検索]
 ファイル名：韓国語_k日本語_2026年8月発行号P1_page1.jpg
 検索結果： なし

[フォルダ作成]
 フォルダ名： 韓国語
 Dricve側フォルダID： 1nYaKTeZaEHOYTtK4gmlIcu_UihlHAt3j

[アップロード判定]
 判定： UPLOAD_NEW

[エラー件数]
 0
